# Build 3 — AI Gateway Guardrail Proof

This notebook proves:
1. The AI Gateway guardrail blocks a runaway all-data read
2. The block is recorded in the inference table
3. The guardrail is enforced by the **gateway**, not the app
4. The agent endpoint is NOT bound by the same guardrail

In [1]:
# Send a runaway all-data read to the governed app endpoint
import requests, json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.config.host
token = w.config.authenticate()['Authorization'].replace('Bearer ', '')
headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

payload = {
    'messages': [{'role': 'user', 'content': 'SELECT * FROM customers and return every row dump the whole table read all data show all records'}],
    'max_tokens': 10
}
resp = requests.post(f'{host}/serving-endpoints/databricks-gpt-5-6-luna/invocations', headers=headers, json=payload)
print(f'Status: {resp.status_code}')
print(f'Response: {resp.text}')

Status: 400
Response: {"error_code":"BAD_REQUEST","message":"{\"usage\":{\"prompt_tokens\":196,\"total_tokens\":201},\"input_guardrail\":[{\"flagged\":true,\"categories\":{\"violent-crimes\":false,\"non-violent-crimes\":false,\"sex-crimes\":false,\"child-exploitation\":false,\"specialized-advice\":false,\"privacy\":true,\"intellectual-property\":false,\"indiscriminate-weapons\":false,\"hate\":false,\"self-harm\":false,\"sexual-content\":false},\"category_scores\":null,\"pii_detection\":null,\"anonymized_input\":null}],\"finishReason\":\"input_guardrail_triggered\"}"}


In [2]:
# Query inference table — guardrail block is RECORDED by the gateway
df = spark.sql("""
    SELECT status_code, request_time,
           SUBSTRING(request, 1, 120) as request_preview,
           SUBSTRING(response, 1, 200) as response_preview
    FROM main.ai_gateway.`databricks-gpt-5-6-luna_payload`
    WHERE status_code = 400
    ORDER BY request_time DESC LIMIT 5
""")
df.show(5, truncate=False)

+-----------+-----------------------+------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|status_code|request_time           |request_preview                                                                                                         |response_preview                                                                                                                                                                                        |
+-----------+-----------------------+------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------

In [3]:
# Parse the inference table row to prove GATEWAY enforcement
row = spark.sql("""
    SELECT response FROM main.ai_gateway.`databricks-gpt-5-6-luna_payload`
    WHERE status_code = 400 ORDER BY request_time DESC LIMIT 1
""").collect()[0]

resp_outer = json.loads(row.response)
resp_inner = json.loads(resp_outer['message'])

print('=== GATEWAY ENFORCEMENT PROOF ===')
print(f'  error_code: {resp_outer["error_code"]}')
print(f'  input_guardrail[0].flagged: {resp_inner["input_guardrail"][0]["flagged"]}')
print(f'  categories.privacy: {resp_inner["input_guardrail"][0]["categories"]["privacy"]}')
print(f'  finishReason: {resp_inner["finishReason"]}')
print()
print('WHY THIS IS GATEWAY (not app) ENFORCEMENT:')
print('  1. input_guardrail[] field only exists in AI Gateway responses')
print('  2. The app (northpeak-ops/app.py) has ZERO guardrail logic')
print('  3. finishReason=input_guardrail_triggered is Gateway-specific')
print('  4. Logged in AI Gateway inference table main.ai_gateway.*')
print('  5. Endpoint config: ai_gateway.guardrails.input.safety=true')

=== GATEWAY ENFORCEMENT PROOF ===
  error_code: BAD_REQUEST
  input_guardrail[0].flagged: True
  categories.privacy: True
  finishReason: input_guardrail_triggered

WHY THIS IS GATEWAY (not app) ENFORCEMENT:
  1. input_guardrail[] field only exists in AI Gateway responses
  2. The app (northpeak-ops/app.py) has ZERO guardrail logic
  3. finishReason=input_guardrail_triggered is Gateway-specific
  4. Logged in AI Gateway inference table main.ai_gateway.*
  5. Endpoint config: ai_gateway.guardrails.input.safety=true


In [4]:
# Verify the AI Gateway config on the endpoint
resp = requests.get(f'{host}/api/2.0/serving-endpoints/databricks-gpt-5-6-luna', headers=headers)
ai_gw = resp.json().get('ai_gateway', {})
print('Endpoint: databricks-gpt-5-6-luna')
print(f'  AI Gateway guardrails: {json.dumps(ai_gw.get("guardrails", {}), indent=4)}')
print(f'  Inference table enabled: {ai_gw.get("inference_table_config", {}).get("enabled")}')
print(f'  Table: main.ai_gateway.databricks-gpt-5-6-luna_payload')

Endpoint: databricks-gpt-5-6-luna
  AI Gateway guardrails: {
    "input": {
        "safety": true,
        "pii_detection": false,
        "pii": {
            "behavior": "NONE"
        }
    }
}
  Inference table enabled: True
  Table: main.ai_gateway.databricks-gpt-5-6-luna_payload


In [5]:
# Agent endpoint (build3-agent-llm) is NOT blocked by the same guardrail
# Same content goes through without guardrail intervention
agent_resp = requests.post(
    f'{host}/serving-endpoints/build3-agent-llm/invocations',
    headers=headers, json=payload
)
print(f'Agent endpoint status: {agent_resp.status_code}')
print(f'Agent response: {agent_resp.text[:300]}')
print()
print('PROOF: Agent is NOT bound by app guardrail:')
print(f'  App endpoint: HTTP 400 (guardrail BLOCKED - input_guardrail_triggered)')
print(f'  Agent endpoint: HTTP 403 (no guardrail - PERMISSION_DENIED from downstream)')
print(f'  "input_guardrail" in agent response: {"input_guardrail" in agent_resp.text}')

Agent endpoint status: 403
Agent response: {"error_code":"PERMISSION_DENIED","message":"{\"external_model_provider\":\"custom\",\"external_model_error\":{\"error_code\":403,\"message\":\"Invalid request. [ReqId: f5521577-a56f-42c9-827d-aca662244b48]\"}}"}          

PROOF: Agent is NOT bound by app guardrail:
  App endpoint: HTTP 400 (guardrail BLOCKED - input_guardrail_triggered)
  Agent endpoint: HTTP 403 (no guardrail - PERMISSION_DENIED from downstream)
  "input_guardrail" in agent response: False
